# SmartCart – Exploratory Data Analysis (EDA)

## Project: Global E-Commerce Orders & Returns Analysis

### Objective
To analyze cleaned e-commerce datasets, identify business patterns, explore customer and product performance, and answer important business questions using data.

### Analysis Areas
1. Sales and Revenue Analysis
2. Customer Behaviour Analysis
3. Product Performance Analysis
4. Profitability Analysis
5. Returns and Refund Analysis
6. Country and Regional Performance

### Tools & Libraries
- Python
- Pandas
- NumPy
- Matplotlib
- Seaborn

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

In [2]:
project_path = Path.cwd().parent
processed_path = project_path / "Data" / "processed"

print("Project folder:", project_path)
print("Processed data folder:", processed_path)

Project folder: c:\Users\Aditya\Desktop\SmartCart-Analytics
Processed data folder: c:\Users\Aditya\Desktop\SmartCart-Analytics\Data\processed


In [3]:
customers = pd.read_csv(processed_path / "customers_clean.csv")
products = pd.read_csv(processed_path / "products_clean.csv")
orders = pd.read_csv(processed_path / "orders_clean.csv")
order_items = pd.read_csv(processed_path / "order_items_clean.csv")
returns = pd.read_csv(processed_path / "returns_clean.csv")

print("All five cleaned datasets loaded successfully!")

All five cleaned datasets loaded successfully!


In [4]:
datasets = {
    "Customers": customers,
    "Products": products,
    "Orders": orders,
    "Order Items": order_items,
    "Returns": returns
}

for name, df in datasets.items():
    print(f"{name}: {df.shape}")

Customers: (3000, 12)
Products: (345, 10)
Orders: (14000, 21)
Order Items: (29671, 11)
Returns: (2433, 15)


In [5]:
total_orders = orders["order_id"].nunique()

total_revenue = orders["order_total_usd"].sum()

average_order_value = orders["order_total_usd"].mean()

total_units_sold = order_items["quantity"].sum()

print("Total Orders:", total_orders)
print("Total Revenue (USD):", round(total_revenue, 2))
print("Average Order Value (USD):", round(average_order_value, 2))
print("Total Units Sold:", total_units_sold)

Total Orders: 14000
Total Revenue (USD): 7938040.75
Average Order Value (USD): 567.0
Total Units Sold: 41784


In [6]:
status_revenue = (
    orders.groupby("order_status")
    .agg(
        total_orders=("order_id", "nunique"),
        total_revenue_usd=("order_total_usd", "sum")
    )
    .reset_index()
    .sort_values("total_revenue_usd", ascending=False)
)

status_revenue

,order_status,total_orders,total_revenue_usd
1,Delivered,12030,6842340.19
0,Cancelled,807,437552.35
3,Shipped,724,423027.25
2,Processing,439,235120.96


In [7]:
delivered_orders = orders[
    orders["order_status"] == "Delivered"
]

delivered_revenue = delivered_orders["order_total_usd"].sum()

delivery_rate = (
    len(delivered_orders) / len(orders)
) * 100

print("Delivered Revenue (USD):", round(delivered_revenue, 2))
print("Delivery Rate (%):", round(delivery_rate, 2))

Delivered Revenue (USD): 6842340.19
Delivery Rate (%): 85.93


### Sales Analysis – Initial Findings

- Total orders: 14,000
- Total recorded order value: $7,938,040.75
- Average order value: $567.00
- Total units sold: 41,784
- Delivered orders: 12,030
- Delivered revenue: $6,842,340.19
- Delivery rate: 85.93%

Key Insight:
Delivered orders account for approximately 86.2% of total recorded order value. Cancelled, shipped, and processing orders should be considered separately when reporting completed sales.

In [8]:
channel_analysis = (
    orders.groupby("sales_channel")
    .agg(
        total_orders=("order_id", "nunique"),
        revenue_usd=("order_total_usd", "sum"),
        average_order_value=("order_total_usd", "mean")
    )
    .reset_index()
    .sort_values("revenue_usd", ascending=False)
)

channel_analysis.round(2)

,sales_channel,total_orders,revenue_usd,average_order_value
1,Mobile App,5872,3340403.67,568.87
3,Web Store,4689,2631828.32,561.28
0,Marketplace,1821,1036262.86,569.06
2,Social Shop,1618,929545.90,574.50


### Sales Channel Analysis – Key Findings

- Mobile App generated the highest revenue ($3.34M) with 5,872 orders.
- Web Store ranked second, generating $2.63M.
- Social Shop recorded the highest average order value ($574.50).
- Marketplace generated $1.04M from 1,821 orders.

Key Insight:
The Mobile App is the largest sales channel by order volume and revenue, while Social Shop has the highest average order value.

In [9]:
orders["order_date_corrected"] = pd.to_datetime(orders["order_date_corrected"])

orders["order_month"] = orders["order_date_corrected"].dt.to_period("M")

print("Date range:", orders["order_date_corrected"].min(), "to", orders["order_date_corrected"].max())

Date range: 2024-01-01 00:00:00 to 2026-06-29 00:00:00


In [10]:
monthly_sales = (
    orders.groupby("order_month")
    .agg(
        total_orders=("order_id", "nunique"),
        revenue_usd=("order_total_usd", "sum")
    )
    .reset_index()
)

monthly_sales["order_month"] = monthly_sales["order_month"].astype(str)

monthly_sales.round(2)

,order_month,total_orders,revenue_usd
0,2024-01,155,88825.60
1,2024-02,146,89083.21
2,2024-03,182,106420.43
3,2024-04,210,117240.15
4,2024-05,207,110941.03
5,2024-06,235,133670.76
6,2024-07,233,137822.62
7,2024-08,284,166606.78
8,2024-09,287,146649.89
9,2024-10,335,191233.11


### Monthly Sales Analysis – Key Findings

- Monthly order volume and recorded order value show an overall upward trend.
- March 2026 recorded the highest monthly revenue of $457,141.42.
- May 2026 recorded the highest order volume with 807 orders.
- February 2024 had the lowest order volume with 146 orders.
- June 2026 is a partial month, covering data only through June 29.

Key Insight:
SmartCart experienced substantial growth in recorded order activity between 2024 and 2026. Monthly comparisons should account for the incomplete June 2026 period.

In [11]:
country_sales = (
    orders.groupby("shipping_country")
    .agg(
        total_orders=("order_id", "nunique"),
        revenue_usd=("order_total_usd", "sum")
    )
    .reset_index()
    .sort_values("revenue_usd", ascending=False)
)

country_sales.round(2)

,shipping_country,total_orders,revenue_usd
11,United States,3172,1600847.48
10,United Kingdom,1463,868700.26
5,India,1374,839267.79
4,Germany,1270,822167.00
3,France,908,573130.28
7,Saudi Arabia,944,554087.35
9,United Arab Emirates,989,523333.84
2,Canada,848,463048.31
1,Brazil,792,456897.71
6,Japan,804,442438.10


### Country-wise Sales Analysis – Key Findings

- The United States generated the highest recorded order value ($1.60M) from 3,172 orders.
- The United Kingdom ranked second with $868,700.26.
- India ranked third with $839,267.79.
- Germany followed with $822,167.00.
- Spain recorded the lowest revenue at $371,194.64.

Key Insight:
The United States is SmartCart's largest market by order volume and recorded order value. India and Germany are also significant markets for the business.

In [12]:
region_sales = (
    orders.groupby("shipping_region")
    .agg(
        total_orders=("order_id", "nunique"),
        revenue_usd=("order_total_usd", "sum")
    )
    .reset_index()
    .sort_values("revenue_usd", ascending=False)
)

region_sales.round(2)

,shipping_region,total_orders,revenue_usd
1,Europe,4285,2635192.18
4,North America,4020,2063895.79
0,Asia-Pacific,2970,1704633.88
3,Middle East,1933,1077421.19
2,Latin America,792,456897.71


### Region-wise Sales Analysis – Key Findings

- Europe generated the highest recorded revenue ($2.64M) from 4,285 orders.
- North America ranked second with $2.06M.
- Asia-Pacific generated $1.70M from 2,970 orders.
- Middle East generated $1.08M.
- Latin America recorded the lowest revenue at $456,897.71.

Key Insight:
Europe is SmartCart's largest regional market, followed by North America and Asia-Pacific. Latin America currently contributes the smallest share of recorded order value.

In [13]:
acquisition_analysis = (
    customers.groupby("acquisition_channel")
    .agg(
        total_customers=("customer_id", "nunique")
    )
    .reset_index()
    .sort_values("total_customers", ascending=False)
)

acquisition_analysis

,acquisition_channel,total_customers
4,Organic Search,676
6,Paid Social,569
5,Paid Search,522
2,Email,424
1,Direct,347
0,Affiliate,244
3,Marketplace,218


### Customer Acquisition Analysis – Key Findings

- Organic Search acquired the highest number of customers (676).
- Paid Social ranked second with 569 customers.
- Paid Search contributed 522 customers.
- Email brought 424 customers.
- Marketplace recorded the lowest customer count (218).

Key Insight:
Organic Search is SmartCart's largest customer acquisition channel, while Paid Social and Paid Search are also important sources of customer acquisition.

In [14]:
customers.columns.tolist()

['customer_id',
 'customer_name',
 'country',
 'region',
 'currency_code',
 'signup_date',
 'acquisition_channel',
 'preferred_device',
 'newsletter_opt_in',
 'first_order_date',
 'lifetime_orders',
 'lifetime_spend_usd']

In [15]:
device_analysis = (
    customers.groupby("preferred_device")
    .agg(
        total_customers=("customer_id", "nunique")
    )
    .reset_index()
    .sort_values("total_customers", ascending=False)
)

device_analysis

,preferred_device,total_customers
1,Mobile,1708
0,Desktop,1028
2,Tablet,264


### Customer Device Analysis – Key Findings

- Mobile is the most preferred device, used by 1,708 customers (56.93%).
- Desktop ranks second with 1,028 customers (34.27%).
- Tablet is the least preferred device, used by 264 customers (8.80%).

Key Insight:
More than half of SmartCart's customers prefer mobile devices. Improving the mobile shopping experience could help support customer engagement and conversions.

In [16]:
newsletter_analysis = (
    customers.groupby("newsletter_opt_in")
    .agg(
        total_customers=("customer_id", "nunique")
    )
    .reset_index()
)

newsletter_analysis

,newsletter_opt_in,total_customers
0,No,1653
1,Yes,1347


### Newsletter Subscription Analysis – Key Findings

- 1,653 customers (55.1%) have not opted into newsletters.
- 1,347 customers (44.9%) have opted in.

Key Insight:
More than half of SmartCart's customers have not opted into newsletters. The business could explore ways to improve newsletter engagement while respecting customer communication preferences.

In [17]:
clv_analysis = customers["lifetime_spend_usd"].agg(
    total_lifetime_revenue="sum",
    average_clv="mean",
    median_clv="median",
    minimum_clv="min",
    maximum_clv="max"
)

clv_analysis.round(2)

total_lifetime_revenue    7500488.40
average_clv                  2500.16
median_clv                   2166.69
minimum_clv                     0.00
maximum_clv                 11028.75
Name: lifetime_spend_usd, dtype: float64

### Customer Lifetime Value Analysis – Key Findings

- Total customer lifetime spending is $7,500,488.40.
- Average customer lifetime value is $2,500.16.
- Median customer lifetime value is $2,166.69.
- Minimum lifetime spending is $0.00.
- Maximum lifetime spending is $11,028.75.

Key Insight:
Customer lifetime spending varies considerably. SmartCart can investigate high-value customer segments and opportunities to encourage repeat purchases.

In [18]:
purchase_frequency = (
    customers["lifetime_orders"]
    .value_counts()
    .sort_index()
    .reset_index()
)

purchase_frequency.columns = ["lifetime_orders", "total_customers"]

purchase_frequency

,lifetime_orders,total_customers
0,0,34
1,1,158
2,2,368
3,3,514
4,4,569
5,5,539
6,6,348
7,7,221
8,8,140
9,9,73


### Customer Purchase Frequency Analysis – Key Findings

- SmartCart has 3,000 customers in total.
- 34 customers have not placed any orders.
- 2,148 customers have placed between 1 and 5 orders.
- 818 customers have placed 6 or more orders.
- The most common purchase frequency is 4 orders, reported by 569 customers.

Key Insight:
Most customers have made multiple purchases, while a smaller group of frequent buyers has placed six or more orders. SmartCart can study these frequent buyers to understand repeat-purchase behaviour.

In [19]:
acquisition_clv = (
    customers.groupby("acquisition_channel")
    .agg(
        total_customers=("customer_id", "nunique"),
        total_lifetime_spend=("lifetime_spend_usd", "sum"),
        average_clv=("lifetime_spend_usd", "mean")
    )
    .reset_index()
    .sort_values("average_clv", ascending=False)
)

acquisition_clv.round(2)

,acquisition_channel,total_customers,total_lifetime_spend,average_clv
0,Affiliate,244,653695.36,2679.08
2,Email,424,1114634.85,2628.86
3,Marketplace,218,544821.72,2499.18
6,Paid Social,569,1408585.53,2475.55
5,Paid Search,522,1290010.86,2471.29
4,Organic Search,676,1651772.25,2443.45
1,Direct,347,836967.83,2412.01


### Acquisition Channel vs Customer Lifetime Value – Key Findings

- Affiliate has the highest average CLV at $2,679.08.
- Email ranks second with an average CLV of $2,628.86.
- Organic Search brings the highest number of customers (676).
- Direct has the lowest average CLV at $2,412.01.
- Paid Social and Paid Search contribute substantial customer lifetime spending.

Key Insight:
Affiliate and Email acquisition channels attract customers with higher average lifetime spending, while Organic Search contributes the largest customer volume. SmartCart can evaluate both customer acquisition volume and long-term customer value when assessing marketing channels.

In [20]:
customer_country = (
    customers.groupby("country")
    .agg(
        total_customers=("customer_id", "nunique"),
        total_lifetime_spend=("lifetime_spend_usd", "sum"),
        average_clv=("lifetime_spend_usd", "mean")
    )
    .reset_index()
    .sort_values("average_clv", ascending=False)
)

customer_country.round(2)

,country,total_customers,total_lifetime_spend,average_clv
4,Germany,272,771632.92,2836.89
5,India,286,795978.82,2783.14
3,France,200,541929.63,2709.65
10,United Kingdom,310,822343.75,2652.72
8,Spain,136,358677.82,2637.34
7,Saudi Arabia,198,519255.73,2622.50
1,Brazil,174,425957.04,2448.03
2,Canada,180,439577.01,2442.09
6,Japan,177,407142.06,2300.24
11,United States,662,1516547.85,2290.86


### Customer Country Analysis – Key Findings

- Germany has the highest average CLV at $2,836.89.
- India ranks second with an average CLV of $2,783.14.
- The United States has the largest customer base with 662 customers.
- The United Kingdom contributes the highest total lifetime spending ($822,343.75).
- Australia has the lowest average CLV at $2,165.30.

Key Insight:
Customer volume and customer value differ across markets. Germany and India have higher average customer lifetime spending, while the United States leads in customer count. SmartCart can consider both customer volume and lifetime value when evaluating country-level opportunities.

In [21]:
print("Products columns:")
print(products.columns.tolist())

print("\nOrder Items columns:")
print(order_items.columns.tolist())

Products columns:
['product_id', 'product_name', 'category', 'subcategory', 'brand', 'list_price_usd', 'unit_cost_usd', 'weight_kg', 'launch_date', 'is_active']

Order Items columns:
['order_item_id', 'order_id', 'product_id', 'quantity', 'unit_price_local', 'line_discount_local', 'line_total_local', 'line_total_usd', 'unit_cost_usd', 'line_profit_usd', 'currency_code']


In [22]:
product_sales = order_items.merge(
    products[["product_id", "product_name", "category", "subcategory"]],
    on="product_id",
    how="left"
)

print("Merged dataset shape:", product_sales.shape)

product_sales.head()

Merged dataset shape: (29671, 14)


,order_item_id,order_id,product_id,quantity,unit_price_local,line_discount_local,line_total_local,line_total_usd,unit_cost_usd,line_profit_usd,currency_code,product_name,category,subcategory
0,ITM-000001,ORD-000001,PRD-0061,1,42.78,0.00,42.78,42.78,18.99,23.79,USD,Vantia Footwear 0061,Apparel,Footwear
1,ITM-000002,ORD-000001,PRD-0311,1,96.79,0.00,96.79,96.79,53.60,43.19,USD,Kestrel Plush 0311,Toys & Games,Plush
2,ITM-000003,ORD-000002,PRD-0335,1,282.11,0.00,282.11,75.23,39.08,36.15,SAR,Solstice Plush 0335,Toys & Games,Plush
3,ITM-000004,ORD-000003,PRD-0092,2,1122.95,369.42,1876.48,1379.76,632.31,115.14,CAD,Marlowe Camera 0092,Electronics,Cameras
4,ITM-000005,ORD-000003,PRD-0301,1,96.07,31.15,64.92,47.74,35.15,12.59,CAD,Brightfold Building Set 0301,Toys & Games,Building Sets


In [23]:
category_performance = (
    product_sales.groupby("category")
    .agg(
        total_units_sold=("quantity", "sum"),
        total_revenue_usd=("line_total_usd", "sum"),
        total_profit_usd=("line_profit_usd", "sum"),
        total_order_items=("order_item_id", "nunique")
    )
    .reset_index()
    .sort_values("total_revenue_usd", ascending=False)
)

category_performance.round(2)

,category,total_units_sold,total_revenue_usd,total_profit_usd,total_order_items
2,Electronics,9019,3527746.55,793729.68,6346
4,Sports,5530,1172407.47,444626.24,3924
0,Apparel,10881,1072433.35,558842.98,7804
3,Home & Kitchen,5435,623454.39,249683.15,3859
1,Beauty,5494,362591.00,215417.84,3868
5,Toys & Games,5425,327765.41,146057.19,3870


### Product Category Performance – Key Findings

- Electronics generated the highest recorded revenue ($3.53M) and profit ($793.73K).
- Apparel recorded the highest unit sales (10,881) and order-item count (7,804).
- Sports ranked second in recorded revenue ($1.17M).
- Toys & Games recorded the lowest revenue ($327.77K) and profit ($146.06K).

Key Insight:
Electronics is the strongest category by recorded revenue and profit, while Apparel leads in sales volume. Toys & Games may require further product-level investigation.

In [24]:
product_performance = (
    product_sales.groupby(["product_id", "product_name", "category"])
    .agg(
        total_units_sold=("quantity", "sum"),
        total_revenue_usd=("line_total_usd", "sum"),
        total_profit_usd=("line_profit_usd", "sum")
    )
    .reset_index()
    .sort_values("total_revenue_usd", ascending=False)
)

product_performance.head(10).round(2)

,product_id,product_name,category,total_units_sold,total_revenue_usd,total_profit_usd
104,PRD-0105,Corvid Wearable 0105,Electronics,147,106254.78,21347.58
117,PRD-0118,Corvid Headphone 0118,Electronics,130,106002.98,27662.38
92,PRD-0093,Marlowe Wearable 0093,Electronics,132,105228.42,29620.14
111,PRD-0112,Vantia Headphone 0112,Electronics,135,100989.54,17195.04
131,PRD-0132,Auralis Camera 0132,Electronics,127,99319.79,23268.38
96,PRD-0097,Brightfold Wearable 0097,Electronics,119,99180.42,26772.48
93,PRD-0094,Kestrel Camera 0094,Electronics,123,94169.39,23227.91
158,PRD-0159,Northline Wearable 0159,Electronics,122,92886.01,18622.17
91,PRD-0092,Marlowe Camera 0092,Electronics,117,90319.36,16339.09
122,PRD-0123,Solstice Wearable 0123,Electronics,110,86144.03,14652.83


### Top 10 Products by Revenue – Key Findings

- Corvid Wearable 0105 generated the highest recorded revenue ($106,254.78).
- Corvid Headphone 0118 ranked second ($106,002.98).
- Marlowe Wearable 0093 generated the highest profit among the top 10 ($29,620.14).
- All 10 products belong to the Electronics category.
- Wearables, headphones, and cameras appear frequently among the top performers.

Key Insight:
Electronics' strong category performance is supported by multiple high-revenue products across different product types.

In [25]:
top_profitable_products = (
    product_performance
    .sort_values("total_profit_usd", ascending=False)
    .head(10)
)

top_profitable_products.round(2)

,product_id,product_name,category,total_units_sold,total_revenue_usd,total_profit_usd
92,PRD-0093,Marlowe Wearable 0093,Electronics,132,105228.42,29620.14
117,PRD-0118,Corvid Headphone 0118,Electronics,130,106002.98,27662.38
96,PRD-0097,Brightfold Wearable 0097,Electronics,119,99180.42,26772.48
131,PRD-0132,Auralis Camera 0132,Electronics,127,99319.79,23268.38
93,PRD-0094,Kestrel Camera 0094,Electronics,123,94169.39,23227.91
289,PRD-0290,Corvid Cycling 0290,Sports,135,53197.28,21867.83
104,PRD-0105,Corvid Wearable 0105,Electronics,147,106254.78,21347.58
281,PRD-0282,Auralis Fitnes 0282,Sports,145,53470.32,21329.62
145,PRD-0146,Ferrix Camera 0146,Electronics,150,70515.83,20597.33
132,PRD-0133,Ferrix Headphone 0133,Electronics,97,80038.60,20295.33


### Top 10 Most Profitable Products – Key Findings

- Marlowe Wearable 0093 generated the highest profit ($29,620.14).
- Corvid Headphone 0118 ranked second ($27,662.38).
- 8 of the top 10 most profitable products belong to Electronics.
- Two Sports products also appear in the top 10.
- Corvid Wearable 0105 leads revenue but ranks seventh by profit.

Key Insight:
High revenue does not always mean the highest profitability. Product decisions should consider both revenue and profit.

In [26]:
lowest_profit_products = (
    product_performance
    .sort_values("total_profit_usd", ascending=True)
    .head(10)
)

lowest_profit_products.round(2)

,product_id,product_name,category,total_units_sold,total_revenue_usd,total_profit_usd
202,PRD-0203,Tanuki Lighting 0203,Home & Kitchen,127,1466.15,582.23
177,PRD-0178,Northline Lighting 0178,Home & Kitchen,98,1578.64,603.54
242,PRD-0243,Vantia Fragrance 0243,Beauty,108,1060.43,615.47
174,PRD-0175,Vantia Storage 0175,Home & Kitchen,107,1805.92,683.49
191,PRD-0192,Corvid Textile 0192,Home & Kitchen,114,1752.00,705.48
338,PRD-0339,Tanuki Board Game 0339,Toys & Games,111,1875.27,794.13
264,PRD-0265,Vantia Fitnes 0265,Sports,120,2036.23,823.03
175,PRD-0176,Northline Lighting 0176,Home & Kitchen,124,2518.93,913.13
318,PRD-0319,Brightfold Board Game 0319,Toys & Games,124,1903.89,940.41
65,PRD-0066,Corvid Jacket 0066,Apparel,115,1954.85,952.04


### Lowest-Profit Products – Key Findings

- Tanuki Lighting 0203 recorded the lowest total profit ($582.23).
- Northline Lighting 0178 ranked second-lowest ($603.54).
- Home & Kitchen contributed 5 products to the bottom 10.
- Toys & Games contributed 2 products.
- All 10 products generated positive total profit.

Key Insight:
Home & Kitchen appears frequently among the lowest-profit products. Pricing, costs, and discount levels may need further investigation.

In [27]:
product_performance["profit_margin_pct"] = (
    product_performance["total_profit_usd"]
    / product_performance["total_revenue_usd"]
) * 100

product_performance.sort_values(
    "profit_margin_pct", ascending=False
).head(10).round(2)

,product_id,product_name,category,total_units_sold,total_revenue_usd,total_profit_usd,profit_margin_pct
211,PRD-0212,Brightfold Skincare 0212,Beauty,157,4004.10,2526.73,63.10
239,PRD-0240,Northline Fragrance 0240,Beauty,146,8446.98,5281.70,62.53
222,PRD-0223,Auralis Skincare 0223,Beauty,143,3425.97,2136.11,62.35
238,PRD-0239,Marlowe Haircare 0239,Beauty,133,13055.38,8114.43,62.15
232,PRD-0233,Tanuki Skincare 0233,Beauty,110,11257.87,6995.37,62.14
235,PRD-0236,Tanuki Skincare 0236,Beauty,89,9149.71,5684.94,62.13
240,PRD-0241,Auralis Fragrance 0241,Beauty,135,11502.13,7140.28,62.08
231,PRD-0232,Kestrel Fragrance 0232,Beauty,150,13575.05,8425.55,62.07
234,PRD-0235,Northline Fragrance 0235,Beauty,118,6971.72,4324.98,62.04
248,PRD-0249,Corvid Fragrance 0249,Beauty,119,13548.57,8368.50,61.77


### Highest Profit-Margin Products – Key Findings

- Brightfold Skincare 0212 recorded the highest profit margin (63.10%).
- Northline Fragrance 0240 ranked second (62.53%).
- All 10 products with the highest profit margins belong to Beauty.
- Beauty products demonstrate strong margins despite lower category-level revenue than Electronics.

Key Insight:
Beauty products show strong profit-margin potential. The category's overall profitability should be compared with its sales volume and revenue before making business decisions.

In [28]:
category_margin = (
    product_sales.groupby("category")
    .agg(
        total_revenue_usd=("line_total_usd", "sum"),
        total_profit_usd=("line_profit_usd", "sum")
    )
    .reset_index()
)

category_margin["profit_margin_pct"] = (
    category_margin["total_profit_usd"]
    / category_margin["total_revenue_usd"]
) * 100

category_margin.sort_values(
    "profit_margin_pct", ascending=False
).round(2)

,category,total_revenue_usd,total_profit_usd,profit_margin_pct
1,Beauty,362591.00,215417.84,59.41
0,Apparel,1072433.35,558842.98,52.11
5,Toys & Games,327765.41,146057.19,44.56
3,Home & Kitchen,623454.39,249683.15,40.05
4,Sports,1172407.47,444626.24,37.92
2,Electronics,3527746.55,793729.68,22.50


### Category-wise Profit Margin – Key Findings

- Beauty recorded the highest profit margin (59.41%).
- Apparel ranked second with a 52.11% margin.
- Electronics generated the highest total profit ($793,729.68), but had the lowest profit margin (22.50%).
- Sports recorded a 37.92% margin.
- Toys & Games and Home & Kitchen recorded margins of 44.56% and 40.05%, respectively.

Key Insight:
Electronics leads in total profit because of its high revenue, while Beauty leads in profitability relative to revenue. Both metrics are important when evaluating category performance.

In [29]:
discount_analysis = (
    product_sales.groupby("category")
    .agg(
        total_discount_local=("line_discount_local", "sum"),
        total_units_sold=("quantity", "sum"),
        total_revenue_usd=("line_total_usd", "sum"),
        total_profit_usd=("line_profit_usd", "sum")
    )
    .reset_index()
    .sort_values("total_discount_local", ascending=False)
)

discount_analysis.round(2)


,category,total_discount_local,total_units_sold,total_revenue_usd,total_profit_usd
2,Electronics,4456354.67,9019,3527746.55,793729.68
0,Apparel,1494670.53,10881,1072433.35,558842.98
4,Sports,1471385.66,5530,1172407.47,444626.24
3,Home & Kitchen,790865.93,5435,623454.39,249683.15
1,Beauty,467289.54,5494,362591.00,215417.84
5,Toys & Games,414439.65,5425,327765.41,146057.19


In [30]:
product_sales["discount_pct"] = np.where(
    product_sales["unit_price_local"] * product_sales["quantity"] > 0,
    (
        product_sales["line_discount_local"]
        / (product_sales["unit_price_local"] * product_sales["quantity"])
    ) * 100,
    0
)

product_sales["discount_pct"].describe().round(2)

count    29671.00
mean         6.42
std         10.52
min          0.00
25%          0.00
50%          0.00
75%         11.71
max         34.99
Name: discount_pct, dtype: float64

### Discount Analysis – Overall Findings

- Average item-level discount: 6.42%.
- Median discount: 0%.
- 25th percentile: 0%.
- 75th percentile: 11.71%.
- Maximum discount: 34.99%.

Key Insight:
At least half of the order items received no discount. The average discount was 6.42%, suggesting that discounts were not applied uniformly across all purchases.

In [31]:
category_discount = (
    product_sales.groupby("category")
    .agg(
        average_discount_pct=("discount_pct", "mean"),
        median_discount_pct=("discount_pct", "median"),
        total_order_items=("order_item_id", "nunique")
    )
    .reset_index()
    .sort_values("average_discount_pct", ascending=False)
)

category_discount.round(2)

,category,average_discount_pct,median_discount_pct,total_order_items
5,Toys & Games,6.48,0.0,3870
3,Home & Kitchen,6.45,0.0,3859
4,Sports,6.43,0.0,3924
0,Apparel,6.42,0.0,7804
1,Beauty,6.40,0.0,3868
2,Electronics,6.39,0.0,6346


### Category-wise Discount Analysis – Key Findings

- Toys & Games recorded the highest average discount (6.48%).
- Home & Kitchen followed at 6.45%.
- Electronics recorded the lowest average discount (6.39%).
- All six categories had a median discount of 0%.

Key Insight:
Average discount percentages are very similar across categories. Discounting does not appear to vary substantially by product category.

In [32]:
returns.columns.tolist()

['return_id',
 'order_id',
 'order_item_id',
 'product_id',
 'customer_id',
 'return_date',
 'return_month',
 'days_to_return',
 'quantity_returned',
 'return_reason',
 'item_condition',
 'refund_amount_local',
 'refund_amount_usd',
 'restocked',
 'currency_code']

In [33]:
total_returns = returns["return_id"].nunique()
total_returned_units = returns["quantity_returned"].sum()
total_refunds = returns["refund_amount_usd"].sum()
average_refund = returns["refund_amount_usd"].mean()
average_return_days = returns["days_to_return"].mean()

print("Total Return Records:", total_returns)
print("Total Units Returned:", total_returned_units)
print("Total Refund Amount (USD):", round(total_refunds, 2))
print("Average Refund (USD):", round(average_refund, 2))
print("Average Days to Return:", round(average_return_days, 2))

Total Return Records: 2433
Total Units Returned: 2882
Total Refund Amount (USD): 410427.05
Average Refund (USD): 168.69
Average Days to Return: 15.94


### Returns Analysis – Overall Findings

- Total return records: 2,433.
- Total units returned: 2,882.
- Total refund amount: $410,427.05.
- Average refund per return record: $168.69.
- Average time to return: 15.94 days.

Key Insight:
SmartCart recorded over $410K in refunds across 2,433 return records. The average return time was approximately 16 days.

In [34]:
return_reason_analysis = (
    returns.groupby("return_reason")
    .agg(
        total_return_records=("return_id", "nunique"),
        total_units_returned=("quantity_returned", "sum"),
        total_refunds_usd=("refund_amount_usd", "sum")
    )
    .reset_index()
    .sort_values("total_return_records", ascending=False)
)

return_reason_analysis.round(2)

,return_reason,total_return_records,total_units_returned,total_refunds_usd
5,Wrong size or fit,719,838,116858.82
3,Not as described,487,573,76883.91
0,Changed mind,409,499,75326.77
1,Damaged on arrival,312,372,53315.92
4,Quality below expectation,302,361,53355.87
2,Late delivery,204,239,34685.76


### Return Reasons – Key Findings

- Wrong size or fit was the leading return reason, with 719 records (29.6% of all returns).
- It generated the highest refunds ($116,858.82).
- Not as described ranked second with 487 records.
- Changed mind ranked third with 409 records.
- Late delivery was the least common return reason, with 204 records.

Key Insight:
Wrong size or fit is the biggest return-related issue. Better size guides and more accurate product descriptions may help reduce avoidable returns.

In [35]:
returns_product = returns.merge(
    products[["product_id", "product_name", "category", "subcategory"]],
    on="product_id",
    how="left"
)

print("Merged returns dataset shape:", returns_product.shape)

returns_product.head()

Merged returns dataset shape: (2433, 18)


,return_id,order_id,order_item_id,product_id,customer_id,return_date,return_month,days_to_return,quantity_returned,return_reason,item_condition,refund_amount_local,refund_amount_usd,restocked,currency_code,product_name,category,subcategory
0,RET-000001,ORD-003001,ITM-006348,PRD-0016,CST-00837,2024-01-30,2024-01-01,28,1,Changed mind,Damaged,12720.96,152.90,No,INR,Solstice Footwear 0016,Apparel,Footwear
1,RET-000002,ORD-013740,ITM-029103,PRD-0049,CST-01069,2024-01-11,2024-01-01,9,1,Not as described,Damaged,592.48,161.44,Yes,AED,Vantia Jacket 0049,Apparel,Jackets
2,RET-000003,ORD-013740,ITM-029104,PRD-0158,CST-01069,2024-01-10,2024-01-01,8,1,Changed mind,Damaged,122.89,33.49,Yes,AED,Tanuki Camera 0158,Electronics,Cameras
3,RET-000004,ORD-005198,ITM-010985,PRD-0011,CST-00443,2024-01-17,2024-01-01,13,1,Wrong size or fit,Damaged,13157.86,158.15,Yes,INR,Northline Footwear 0011,Apparel,Footwear
4,RET-000005,ORD-009829,ITM-020765,PRD-0208,CST-00528,2024-01-31,2024-01-01,24,1,Changed mind,Resellable,228.00,228.00,No,USD,Solstice Storage 0208,Home & Kitchen,Storage


In [36]:
category_returns = (
    returns_product.groupby("category")
    .agg(
        total_return_records=("return_id", "nunique"),
        total_units_returned=("quantity_returned", "sum"),
        total_refunds_usd=("refund_amount_usd", "sum")
    )
    .reset_index()
    .sort_values("total_return_records", ascending=False)
)

category_returns.round(2)

,category,total_return_records,total_units_returned,total_refunds_usd
0,Apparel,1214,1435,142470.53
4,Sports,342,410,87195.93
2,Electronics,259,312,120775.35
3,Home & Kitchen,221,257,30936.77
5,Toys & Games,206,250,15001.70
1,Beauty,191,218,14046.77


### Category-wise Returns – Key Findings

- Apparel recorded the highest returns: 1,214 records and 1,435 units.
- Electronics generated the highest refunds ($120,775.35).
- Sports ranked second in return records with 342.
- Beauty recorded the fewest returns (191) and lowest refunds ($14,046.77).

Key Insight:
Apparel leads in return volume, while Electronics leads in refund value. Both return frequency and financial impact should be monitored.

In [37]:
category_return_rate = category_performance[
    ["category", "total_units_sold"]
].merge(
    category_returns[
        ["category", "total_units_returned"]
    ],
    on="category",
    how="left"
)

category_return_rate["return_rate_pct"] = (
    category_return_rate["total_units_returned"]
    / category_return_rate["total_units_sold"]
) * 100

category_return_rate.sort_values(
    "return_rate_pct", ascending=False
).round(2)

,category,total_units_sold,total_units_returned,return_rate_pct
2,Apparel,10881,1435,13.19
1,Sports,5530,410,7.41
3,Home & Kitchen,5435,257,4.73
5,Toys & Games,5425,250,4.61
4,Beauty,5494,218,3.97
0,Electronics,9019,312,3.46


### Category-wise Return Rate – Key Findings

- Apparel recorded the highest return rate (13.19%).
- Sports ranked second with a 7.41% return rate.
- Home & Kitchen recorded 4.73%.
- Toys & Games recorded 4.61%.
- Beauty recorded 3.97%.
- Electronics had the lowest return rate (3.46%).

Key Insight:
Apparel has a substantially higher return rate than other categories. Improving size guides, fit information, and product descriptions could help address this issue.

In [38]:
category_reason_analysis = (
    returns_product.groupby(["category", "return_reason"])
    .agg(
        total_return_records=("return_id", "nunique"),
        total_units_returned=("quantity_returned", "sum")
    )
    .reset_index()
    .sort_values(
        ["category", "total_return_records"],
        ascending=[True, False]
    )
)

category_reason_analysis

,category,return_reason,total_return_records,total_units_returned
5,Apparel,Wrong size or fit,340,394
3,Apparel,Not as described,265,319
0,Apparel,Changed mind,184,223
1,Apparel,Damaged on arrival,164,188
4,Apparel,Quality below expectation,156,189
2,Apparel,Late delivery,105,122
11,Beauty,Wrong size or fit,63,76
6,Beauty,Changed mind,39,43
9,Beauty,Not as described,33,35
7,Beauty,Damaged on arrival,25,29


### Return Reasons by Category – Key Findings

- Wrong size or fit was the leading return reason across all six categories.
- Apparel recorded the highest number of wrong-size returns (340).
- Sports followed with 114 wrong-size returns.
- Not as described was another important reason, particularly in Apparel (265 records).
- Changed mind ranked second in Beauty, Electronics, and Sports.

Key Insight:
Wrong size or fit is a consistent return-related issue across all product categories. Improving product specifications, sizing information, and customer guidance may help reduce returns.

In [39]:
return_time_analysis = returns["days_to_return"].agg(
    average_return_days="mean",
    median_return_days="median",
    minimum_return_days="min",
    maximum_return_days="max"
)

return_time_analysis.round(2)

average_return_days    15.94
median_return_days     16.00
minimum_return_days     2.00
maximum_return_days    30.00
Name: days_to_return, dtype: float64

### Step 38 – Return Time Analysis

- Average return time: 15.94 days.
- Median return time: 16 days.
- Minimum return time: 2 days.
- Maximum return time: 30 days.

**Key Insights:**
- Customers return products after approximately 16 days on average.
- The average and median are very close, indicating a fairly balanced distribution around the center.
- Return times range from 2 to 30 days.

In [40]:
return_time_groups = pd.cut(
    returns["days_to_return"],
    bins=[0, 7, 14, 21, 30],
    labels=["1–7 Days", "8–14 Days", "15–21 Days", "22–30 Days"]
)

return_timing_analysis = (
    return_time_groups.value_counts()
    .sort_index()
    .reset_index()
)

return_timing_analysis.columns = [
    "return_time_group",
    "total_returns"
]

return_timing_analysis

,return_time_group,total_returns
0,1–7 Days,520
1,8–14 Days,577
2,15–21 Days,563
3,22–30 Days,773


### Step 39 – Return Timing Groups

- 1–7 days: 520 returns.
- 8–14 days: 577 returns.
- 15–21 days: 563 returns.
- 22–30 days: 773 returns.

**Key Insights:**
- The 22–30 day period records the highest number of returns (31.8%).
- Returns are relatively evenly distributed across the first three periods.
- The business should investigate factors contributing to returns near the end of the return window.

In [41]:
condition_analysis = (
    returns.groupby("item_condition")
    .agg(
        total_return_records=("return_id", "nunique"),
        total_units_returned=("quantity_returned", "sum"),
        total_refunds_usd=("refund_amount_usd", "sum")
    )
    .reset_index()
    .sort_values("total_return_records", ascending=False)
)

condition_analysis.round(2)

,item_condition,total_return_records,total_units_returned,total_refunds_usd
2,Resellable,1501,1760,249158.55
0,Damaged,565,677,100326.63
1,Opened,367,445,60941.87


### Step 40 – Item Condition Analysis

- Resellable: 1,501 records, 1,760 units.
- Damaged: 565 records, 677 units.
- Opened: 367 records, 445 units.

**Key Insights:**
- Resellable items account for approximately 61.7% of return records.
- Damaged items account for approximately 23.2%.
- Resellable items have the highest total refunds due to their larger return volume.
- Damaged and opened items may require separate inspection and restocking policies.

In [42]:
restock_analysis = (
    returns.groupby("restocked")
    .agg(
        total_return_records=("return_id", "nunique"),
        total_units_returned=("quantity_returned", "sum"),
        total_refunds_usd=("refund_amount_usd", "sum"),
        average_refund_usd=("refund_amount_usd", "mean")
    )
    .reset_index()
)

restock_analysis.round(2)

,restocked,total_return_records,total_units_returned,total_refunds_usd,average_refund_usd
0,No,901,1075,158334.05,175.73
1,Yes,1532,1807,252093.00,164.55


### Step 41 – Restocking Analysis

- Restocked: 1,532 records and 1,807 units.
- Not restocked: 901 records and 1,075 units.
- Average refund for restocked items: $164.55.
- Average refund for non-restocked items: $175.73.

**Key Insights:**
- Approximately 63% of return records were marked as restocked.
- Non-restocked returns have a higher average refund.
- Restocked returns account for the larger total refund amount due to their higher volume.

In [43]:
customer_return_analysis = (
    returns.groupby("customer_id")
    .agg(
        total_returns=("return_id", "nunique"),
        total_units_returned=("quantity_returned", "sum"),
        total_refunds_usd=("refund_amount_usd", "sum")
    )
    .reset_index()
)

print("Customers with returns:", customer_return_analysis["customer_id"].nunique())
print("Average returns per returning customer:", round(customer_return_analysis["total_returns"].mean(), 2))

customer_return_analysis.head()

Customers with returns: 1609
Average returns per returning customer: 1.51


,customer_id,total_returns,total_units_returned,total_refunds_usd
0,CST-00001,4,5,629.94
1,CST-00002,1,1,176.43
2,CST-00003,3,3,267.50
3,CST-00008,2,2,460.83
4,CST-00009,2,4,549.03


### Step 42 – Customer Return Analysis

- Customers with at least one return: 1,609.
- Average returns per returning customer: 1.51.
- Returning customers represent approximately 53.6% of the customer base.

**Key Insights:**
- More than half of customers have at least one return.
- Returning customers average 1.51 return records.
- The business should investigate customer experience factors contributing to repeat returns.

In [44]:
return_frequency = (
    customer_return_analysis["total_returns"]
    .value_counts()
    .sort_index()
    .reset_index()
)

return_frequency.columns = [
    "number_of_returns",
    "total_customers"
]

return_frequency

,number_of_returns,total_customers
0,1,1026
1,2,399
2,3,138
3,4,37
4,5,7
5,6,2


### Step 43 – Return Frequency Distribution

- 1 return: 1,026 customers.
- 2 returns: 399 customers.
- 3 returns: 138 customers.
- 4 returns: 37 customers.
- 5 returns: 7 customers.
- 6 returns: 2 customers.

**Key Insights:**
- Approximately 63.8% of returning customers returned only once.
- Around 36.2% returned more than once.
- Only 39 customers had four or more return records.
- Repeat-return customers may be useful to investigate for potential customer experience issues.


In [45]:
customer_return_channels = customer_return_analysis.merge(
    customers[["customer_id", "acquisition_channel"]],
    on="customer_id",
    how="left"
)

channel_return_analysis = (
    customer_return_channels.groupby("acquisition_channel")
    .agg(
        customers_with_returns=("customer_id", "nunique"),
        total_return_records=("total_returns", "sum"),
        total_refunds_usd=("total_refunds_usd", "sum"),
        average_returns_per_customer=("total_returns", "mean")
    )
    .reset_index()
    .sort_values("total_return_records", ascending=False)
)

channel_return_analysis.round(2)

,acquisition_channel,customers_with_returns,total_return_records,total_refunds_usd,average_returns_per_customer
4,Organic Search,344,513,86464.26,1.49
5,Paid Search,291,445,72238.91,1.53
6,Paid Social,301,445,77140.51,1.48
2,Email,220,341,60752.93,1.55
1,Direct,193,298,48858.19,1.54
0,Affiliate,146,216,37648.06,1.48
3,Marketplace,114,175,27324.19,1.54


### Step 44 – Return Analysis by Customer Acquisition Channel

**Key Insights:**
- Organic Search has the highest number of returning customers (344) and return records (513).
- Email has the highest average returns per returning customer (1.55).
- Paid Search and Paid Social each have 445 return records.
- Average returns per returning customer are relatively similar across channels.

**Business Note:**
These figures describe customers who made returns. To compare actual return rates by acquisition channel, the total number of customers in each channel must also be considered.

In [46]:
customer_return_country = customer_return_analysis.merge(
    customers[["customer_id", "country"]],
    on="customer_id",
    how="left"
)

country_return_analysis = (
    customer_return_country.groupby("country")
    .agg(
        customers_with_returns=("customer_id", "nunique"),
        total_return_records=("total_returns", "sum"),
        total_refunds_usd=("total_refunds_usd", "sum")
    )
    .reset_index()
    .sort_values("total_return_records", ascending=False)
)

country_return_analysis.round(2)

,country,customers_with_returns,total_return_records,total_refunds_usd
11,United States,366,537,86432.81
10,United Kingdom,172,261,41931.87
4,Germany,164,239,43967.89
5,India,143,218,33868.42
9,United Arab Emirates,124,198,34802.23
3,France,109,166,29036.13
7,Saudi Arabia,109,158,28009.50
2,Canada,92,146,26179.00
1,Brazil,91,145,27011.25
0,Australia,93,132,21517.58


### Step 45 – Return Analysis by Country

**Key Insights:**
- The United States leads in returning customers, return records, and total refunds.
- The United Kingdom ranks second in return records.
- Germany has higher total refunds than the UK despite fewer return records.
- Spain has the lowest number of return records.

**Business Note:**
Higher return counts do not necessarily mean a higher return rate. Country-level customer totals should be considered before comparing return performance.

In [47]:
country_customer_totals = (
    customers.groupby("country")
    .agg(
        total_customers=("customer_id", "nunique")
    )
    .reset_index()
)

country_return_rate = country_customer_totals.merge(
    country_return_analysis[
        ["country", "customers_with_returns"]
    ],
    on="country",
    how="left"
)

country_return_rate["customers_with_returns"] = (
    country_return_rate["customers_with_returns"].fillna(0)
)

country_return_rate["customer_return_rate_pct"] = (
    country_return_rate["customers_with_returns"]
    / country_return_rate["total_customers"]
) * 100

country_return_rate.sort_values(
    "customer_return_rate_pct",
    ascending=False
).round(2)

,country,total_customers,customers_with_returns,customer_return_rate_pct
4,Germany,272,164,60.29
9,United Arab Emirates,220,124,56.36
10,United Kingdom,310,172,55.48
11,United States,662,366,55.29
7,Saudi Arabia,198,109,55.05
3,France,200,109,54.50
1,Brazil,174,91,52.30
8,Spain,136,71,52.21
2,Canada,180,92,51.11
0,Australia,185,93,50.27


### Step 46 – Country-wise Customer Return Rate

**Key Insights:**
- Germany has the highest customer return rate (60.29%).
- UAE ranks second at 56.36%.
- Japan has the lowest customer return rate (42.37%).
- India's customer return rate is 50%.
- Germany's rate is approximately 17.92 percentage points higher than Japan's.

**Business Recommendation:**
Investigate product preferences, delivery experience, and return reasons in countries with higher customer return rates to identify opportunities for improvement.

## 6. EDA Summary and Business Findings

In [48]:
channel_profitability = order_items.merge(
    orders[["order_id", "sales_channel"]],
    on="order_id",
    how="left"
)

channel_profit_analysis = (
    channel_profitability.groupby("sales_channel")
    .agg(
        total_revenue_usd=("line_total_usd", "sum"),
        total_profit_usd=("line_profit_usd", "sum"),
        total_units_sold=("quantity", "sum"),
        total_order_items=("order_item_id", "nunique")
    )
    .reset_index()
)

channel_profit_analysis["profit_margin_pct"] = (
    channel_profit_analysis["total_profit_usd"]
    / channel_profit_analysis["total_revenue_usd"]
) * 100

channel_profit_analysis.sort_values(
    "total_profit_usd", ascending=False
).round(2)

,sales_channel,total_revenue_usd,total_profit_usd,total_units_sold,total_order_items,profit_margin_pct
1,Mobile App,2984336.01,1004410.81,17419,12421,33.66
3,Web Store,2348812.14,802076.65,13968,9902,34.15
0,Marketplace,923786.23,317585.30,5528,3926,34.38
2,Social Shop,829463.79,284284.32,4869,3422,34.27


**Key Insights:**
- Mobile App generates the highest revenue ($2,984,336.01) and profit ($1,004,410.81).
- Marketplace has the highest profit margin (34.38%).
- Social Shop generates the lowest total revenue and profit.
- Profit margins are relatively similar across all four sales channels.

**Business Recommendation:**
Continue monitoring Mobile App performance while investigating opportunities to improve profitability across other sales channels.

## 6. EDA Summary and Business Findings

### 1. Sales Performance
- Total recorded orders: 14,000.
- Total recorded order value: $7,938,040.75.
- Delivered order value: $6,842,340.19.
- Mobile App generates the highest recorded revenue among sales channels.
- March 2026 recorded the highest monthly order value.

### 2. Customer Behaviour
- Total customers: 3,000.
- Organic Search is the largest customer acquisition channel.
- 56.93% of customers prefer mobile devices.
- Average customer lifetime spend: $2,500.16.
- Affiliate customers have the highest average lifetime value.

### 3. Product Performance
- Electronics generates the highest recorded revenue and total profit.
- Apparel has the highest units sold.
- Beauty has the highest category profit margin (59.41%).
- All ten highest-margin products belong to Beauty.

### 4. Discount Analysis
- Average discount across order items: 6.42%.
- Median discount: 0%.
- Average discount percentages are similar across categories.

### 5. Returns and Refunds
- Total return records: 2,433.
- Total units returned: 2,882.
- Total refunds: $410,427.05.
- Wrong size or fit is the leading return reason.
- Apparel has the highest category return rate (13.19%).
- The 22–30 day period has the highest number of returns.
- Germany has the highest customer return rate (60.29%).

### 6. Profitability
- Mobile App generates the highest total profit.
- Marketplace has the highest channel profit margin (34.38%).
- Electronics generates the highest category profit.
- Beauty has the highest category profit margin.

### Overall Business Recommendations
1. Improve product sizing information, particularly for Apparel.
2. Investigate product quality and description-related returns.
3. Improve the mobile shopping experience.
4. Study customer behaviour in countries with higher return rates.
5. Monitor product-level profitability and discount strategies.
6. Evaluate channel performance using both revenue and profit margins.

### Conclusion
The exploratory analysis identified important patterns in sales, customer behaviour, product performance, profitability, and returns. These findings will guide the next stage of the SmartCart project: data visualization and business reporting.